# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

# Import Paws again
We use paws again for thsi project

## Install project requirements

Run the next cell once. %pip installs packages into the active notebook kernel. Restart the kernel when installation finishes, then run the notebook from the top.

In [3]:
%pip install -r ../requirements.txt

Ignoring fse: markers 'python_version >= "3.11"' don't match your environment
  Using cached scipy-1.13.1-cp310-cp310-macosx_12_0_arm64.whl.metadata (60 kB)
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/opt/homebrew/Caskroom/miniconda/base/envs/jupyterbook/lib/python3.10/site-packages/python_dateutil-2.9.0.post0.dist-info/METADATA'

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import part1
from datasets import load_dataset

dataset = load_dataset("google-research-datasets/paws", "labeled_final")
paws_outcome_space = part1.clslib.NamedClasses(['not paraphrase', 'paraphrase'])


# Build the same sentence-pair folds used by the earlier experiments.
pairs = list(zip(
    dataset["train"]["sentence1"],
    dataset["train"]["sentence2"],
    dataset["train"]["label"],
))
num_folds = 5
my_cross_validation_sets = [
    part1.experiment.cross_validation_split(pairs, k=num_folds, seed=0)
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# importing embedders

In [ ]:
import sys
import urllib.request
import zipfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import part2
# do not forget to d oablation
GLOVE_DIMS = 100   # 50, 100, 200 or 300
glove_zip = Path("glove.6B.zip")
glove_txt = Path(f"glove.6B.{GLOVE_DIMS}d.txt")

if not glove_txt.exists():
    if not glove_zip.exists():
        print("Downloading GloVe 6B (~822 MB) from the Stanford NLP mirror on Hugging Face...")
        urllib.request.urlretrieve(
            "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip", glove_zip)
    with zipfile.ZipFile(glove_zip) as z:
        z.extract(glove_txt.name)

glove = part2.clslib.PretrainedEmbeddings(str(glove_txt), GLOVE_DIMS)
print(f"vocab size: {len(glove.vocab):,}, dims: {glove.num_dims}")


vocab size: 400,002, dims: 100


In [13]:
class RandomEmbeddings:
    def __init__(self, reference_embeddings, seed=0):
        self.vocab = reference_embeddings.vocab
        self.num_dims = reference_embeddings.num_dims

        rng = np.random.default_rng(seed)
        self.vectors = rng.uniform(
            low=-0.05,
            high=0.05,
            size=(len(self.vocab), self.num_dims),
        ).astype(np.float32)


# mlp
MEan pooling


In [7]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


class DenseDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.as_tensor(features, dtype=torch.float32)
        self.labels = torch.as_tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


class SentencePairMLP(nn.Module):
    """MLP classifier for concatenated GloVe sentence representations."""

    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
            nn.LogSoftmax(dim=-1),
        )

    def forward(self, features):
        return self.network(features)


def glove_sentence(text, tokenizer, glove_model):
    """Mean-pool the GloVe vectors for the tokens in one sentence."""
    token_ids = tokenizer.ids(text)
    if not token_ids:
        return np.zeros(glove_model.num_dims, dtype=np.float32)
    vectors = glove_model.vectors[token_ids]
    return vectors.mean(axis=0).astype(np.float32)


def sentence_pair_features(examples, tokenizer, glove_model):
    """Create [sentence1 embedding; sentence2 embedding] features."""
    return np.stack(
        [
            np.concatenate(
                [
                    glove_sentence(sentence1, tokenizer, glove_model),
                    glove_sentence(sentence2, tokenizer, glove_model),
                ]
            )
            for sentence1, sentence2, _ in examples
        ]
    ).astype(np.float32)


def run_epoch(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = total_correct = total_examples = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            logprobs = model(features)
            loss = criterion(logprobs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            total_correct += (logprobs.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


def cross_validation_train_glove_mlp(
    embedding_model,
    my_cross_validation_sets,
    num_folds,
    num_classes=2,
    hidden_dim=128,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=20,
    device=None,
):
    """Train a sentence-pair MLP using mean-pooled GloVe embeddings."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device(device)

    tokenizer = part2.clslib.BasicTokenizer(
        lowercase=True,
        remove_punct=True,
        remove_stopwords=False,
        vocab=embedding_model.vocab,
    )
    results, models = [], {}

    for fold_index, split in my_cross_validation_sets[0].items():
        training_set = split["training"]
        validation_set = split["validation"]
        X_train = sentence_pair_features(training_set, tokenizer, embedding_model)
        X_validation = sentence_pair_features(validation_set, tokenizer, embedding_model)
        y_train = [int(example[2]) for example in training_set]
        y_validation = [int(example[2]) for example in validation_set]

        train_loader = DataLoader(
            DenseDataset(X_train, y_train), batch_size=batch_size, shuffle=True
        )
        validation_loader = DataLoader(
            DenseDataset(X_validation, y_validation), batch_size=batch_size
        )
        model = SentencePairMLP(
            input_dim=2 * embedding_model.num_dims,
            hidden_dim=hidden_dim,
            num_classes=num_classes,
            dropout=dropout,
        ).to(device)
        criterion = nn.NLLLoss()
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=learning_rate, weight_decay=weight_decay
        )

        best_state = None
        best_metrics = None
        best_epoch = 0
        for epoch in tqdm(range(num_epochs), desc=f"GloVe fold {fold_index + 1}/{num_folds}"):
            train_metrics = run_epoch(model, train_loader, criterion, device, optimizer)
            validation_metrics = run_epoch(model, validation_loader, criterion, device)
            if best_metrics is None or validation_metrics["loss"] < best_metrics["loss"]:
                best_metrics = validation_metrics
                best_epoch = epoch
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }

        model.load_state_dict(best_state)
        models[fold_index] = model
        results.append({
            "fold": fold_index,
            "accuracy": best_metrics["accuracy"],
            "best_epoch": best_epoch,
        })
        print(
            f"Fold {fold_index + 1}/{num_folds}: "
            f"{best_metrics['accuracy']:.4f} "
            f"(best epoch {best_epoch + 1})"
        )

    print(f"Mean accuracy: {np.mean([result['accuracy'] for result in results]):.4f}")
    return results, models


# GloVe baseline: the fold examples already contain sentence pairs and labels.
glove_results, glove_models = cross_validation_train_glove_mlp(
    embedding_model=glove,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    num_classes=len(paws_outcome_space),
    device=device,
)


GloVe fold 1/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 1/5: 0.5813 (best epoch 11)


GloVe fold 2/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 2/5: 0.5751 (best epoch 9)


GloVe fold 3/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 3/5: 0.5745 (best epoch 10)


GloVe fold 4/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 4/5: 0.5659 (best epoch 6)


GloVe fold 5/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 5/5: 0.5847 (best epoch 13)
Mean accuracy: 0.5763


In [15]:

random_embedding_model = RandomEmbeddings(glove)

random_results, random_models = cross_validation_train_glove_mlp(
    embedding_model=random_embedding_model,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    num_classes=len(paws_outcome_space),
    device=device,
)

GloVe fold 1/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 1/5: 0.5649 (best epoch 16)


GloVe fold 2/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 2/5: 0.5681 (best epoch 10)


GloVe fold 3/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 3/5: 0.5613 (best epoch 10)


GloVe fold 4/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 4/5: 0.5601 (best epoch 9)


GloVe fold 5/5:   0%|          | 0/20 [00:00<?, ?it/s]

Fold 5/5: 0.5662 (best epoch 5)
Mean accuracy: 0.5641


# LSTM pooling

In [16]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

class SentencePairSequenceDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.examples = examples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        sentence1, sentence2, label = self.examples[index]
        ids1 = self.tokenizer.ids(sentence1) or [0]
        ids2 = self.tokenizer.ids(sentence2) or [0]
        return (
            torch.tensor(ids1, dtype=torch.long),
            torch.tensor(ids2, dtype=torch.long),
            int(label),
        )


def lstm_collate(batch):
    sequences1, sequences2, labels = zip(*batch)

    lengths1 = torch.tensor([len(x) for x in sequences1], dtype=torch.long)
    lengths2 = torch.tensor([len(x) for x in sequences2], dtype=torch.long)

    sequences1 = pad_sequence(sequences1, batch_first=True, padding_value=0)
    sequences2 = pad_sequence(sequences2, batch_first=True, padding_value=0)
    labels = torch.tensor(labels, dtype=torch.long)

    return sequences1, lengths1, sequences2, lengths2, labels


class SentencePairLSTMMLP(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        hidden_dim=128,
        num_classes=2,
        dropout=0.1,
        bidirectional=True,
        freeze_embeddings=False,
    ):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=freeze_embeddings,
            padding_idx=0,
        )

        self.lstm = nn.LSTM(
            input_size=embedding_matrix.shape[1],
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidirectional,
        )

        output_dim = hidden_dim * (2 if bidirectional else 1)

        self.classifier = nn.Sequential(
            nn.Linear(output_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
            nn.LogSoftmax(dim=-1),
        )

    def encode(self, sequences, lengths):
        embedded = self.embedding(sequences)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, (hidden, _) = self.lstm(packed)

        if self.lstm.bidirectional:
            representation = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            representation = hidden[-1]

        return representation

    def forward(self, sequences1, lengths1, sequences2, lengths2):
        representation1 = self.encode(sequences1, lengths1)
        representation2 = self.encode(sequences2, lengths2)

        features = torch.cat((representation1, representation2), dim=1)
        return self.classifier(features)


def run_lstm_epoch(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for sequences1, lengths1, sequences2, lengths2, labels in loader:
            sequences1 = sequences1.to(device)
            lengths1 = lengths1.to(device)
            sequences2 = sequences2.to(device)
            lengths2 = lengths2.to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad()

            logprobs = model(
                sequences1, lengths1,
                sequences2, lengths2,
            )
            loss = criterion(logprobs, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_correct += (logprobs.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


def cross_validation_train_lstm(
    embedding_model,
    my_cross_validation_sets,
    num_folds,
    hidden_dim=128,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=15,
    freeze_embeddings=False,
    device=None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = part2.clslib.BasicTokenizer(
        lowercase=True,
        remove_punct=True,
        remove_stopwords=False,
        vocab=embedding_model.vocab,
    )

    results = []
    models = {}

    for fold_index, split in my_cross_validation_sets[0].items():
        training_set = split["training"]
        validation_set = split["validation"]

        train_loader = DataLoader(
            SentencePairSequenceDataset(training_set, tokenizer),
            batch_size=batch_size,
            shuffle=True,
            collate_fn=lstm_collate,
        )
        validation_loader = DataLoader(
            SentencePairSequenceDataset(validation_set, tokenizer),
            batch_size=batch_size,
            collate_fn=lstm_collate,
        )

        model = SentencePairLSTMMLP(
            embedding_matrix=embedding_model.vectors,
            hidden_dim=hidden_dim,
            num_classes=len(paws_outcome_space),
            dropout=dropout,
            freeze_embeddings=freeze_embeddings,
        ).to(device)

        criterion = nn.NLLLoss()
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )

        best_state = None
        best_metrics = None
        best_epoch = 0

        for epoch in tqdm(
            range(num_epochs),
            desc=f"LSTM fold {fold_index + 1}/{num_folds}",
        ):
            run_lstm_epoch(
                model, train_loader, criterion, device, optimizer
            )
            validation_metrics = run_lstm_epoch(
                model, validation_loader, criterion, device
            )

            if best_metrics is None or validation_metrics["loss"] < best_metrics["loss"]:
                best_metrics = validation_metrics
                best_epoch = epoch
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }

        model.load_state_dict(best_state)
        models[fold_index] = model

        results.append({
            "fold": fold_index,
            "accuracy": best_metrics["accuracy"],
            "best_epoch": best_epoch,
        })

        print(
            f"Fold {fold_index + 1}/{num_folds}: "
            f"{best_metrics['accuracy']:.4f} "
            f"(best epoch {best_epoch + 1})"
        )

    print(f"Mean accuracy: {np.mean([r['accuracy'] for r in results]):.4f}")
    return results, models


lstm_results, lstm_models = cross_validation_train_lstm(
    embedding_model=glove,
    my_cross_validation_sets=my_cross_validation_sets,
    num_folds=num_folds,
    device=device,
)

LSTM fold 1/5:   0%|          | 0/15 [00:00<?, ?it/s]

KeyboardInterrupt: 